# StandUp4AI Benchmark: Download + F0 Evaluation

Downloads StandUp4AI dataset (3,751 videos, 7 languages) to Google Drive, then evaluates our F0 laughter detector.

**Goal:** Beat their F1=0.51 baseline with our 5-dim F0 features.

## Cell 1: Setup

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/standup4ai')
BASE.mkdir(exist_ok=True)
(BASE / 'audio').mkdir(exist_ok=True)
(BASE / 'features').mkdir(exist_ok=True)
(BASE / 'repo').mkdir(exist_ok=True)

print(f'✅ Drive mounted. Base: {BASE}')

# Install dependencies
subprocess.run(['pip', 'install', '-q', 'yt-dlp', 'librosa', 'soundfile', 'tqdm'])
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

print('✅ Dependencies installed')

## Cell 2: Clone StandUp4AI Repo to Drive

In [ ]:
repo_path = BASE / 'repo' / 'seq-Standup4AI'

if not repo_path.exists():
    print('Cloning StandUp4AI repo to Drive...')
    os.chdir(BASE / 'repo')
    subprocess.run(['git', 'clone', '--depth', '1', 
                    'https://github.com/sofia-callejas/seq-Standup4AI.git'], 
                   check=True)
    print(f'✅ Cloned to {repo_path}')
else:
    print(f'✅ Repo already exists at {repo_path}')

# Load partition
partition = pd.read_csv(repo_path / 'partition.csv')
print(f'\nPartition: {len(partition)} videos')
print(partition['part'].value_counts())
print(f'\nLanguages: {partition["lan"].nunique()}')
print(partition['lan'].value_counts().head(10))

# Save test set video IDs (we'll evaluate on these)
test_videos = partition[partition['part'] == 'test']['fn'].tolist()
val_videos = partition[partition['part'] == 'val']['fn'].tolist()
print(f'\nTest: {len(test_videos)} videos')
print(f'Val: {len(val_videos)} videos')

## Cell 3: Explore Dataset Format

In [ ]:
# Check what's in the dataset directory
dataset_dir = repo_path / 'dataset'
print('=== Dataset languages ===')
for d in sorted(dataset_dir.iterdir()):
    if d.is_dir():
        files = list(d.rglob('*'))
        print(f'  {d.name}: {len(files)} items')

# Check a sample file
print('\n=== Sample dataset structure ===')
for lang_dir in sorted(dataset_dir.iterdir()):
    if lang_dir.is_dir() and lang_dir.name != 'multilingual':
        for subdir in sorted(lang_dir.iterdir()):
            if subdir.is_dir():
                sample_files = list(subdir.glob('*'))[:3]
                print(f'  {lang_dir.name}/{subdir.name}/: {len(list(subdir.glob("*")))} files')
                for f in sample_files:
                    print(f'    {f.name} ({f.stat().st_size} bytes)')
            break  # Just first language
        break

# Load a sample laughter detection file
ld_dir = repo_path / 'laughter_detection' / 'EMNLP'
if ld_dir.exists():
    print(f'\n=== EMNLP laughter_detection ===')
    for item in sorted(ld_dir.iterdir()):
        print(f'  {item.type if hasattr(item, "type") else "?"} {item.name}')

## Cell 4: Download Test Set Audio from YouTube

In [ ]:
# Download audio for test + val videos
videos_to_download = test_videos + val_videos[:50]  # Test + 50 val
print(f'Downloading {len(videos_to_download)} videos to Drive...')

import yt_dlp

ydl_opts = {
    'format': 'bestaudio/best',
    'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'm4a'}],
    'outtmpl': str(BASE / 'audio' / '%(id)s.%(ext)s'),
    'quiet': True,
    'no_warnings': True,
}

downloaded = []
failed = []

for i, vid in enumerate(tqdm(videos_to_download)):
    audio_file = BASE / 'audio' / f'{vid}.m4a'
    if audio_file.exists():
        downloaded.append(vid)
        continue
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([f'https://youtube.com/watch?v={vid}'])
        downloaded.append(vid)
    except Exception as e:
        failed.append((vid, str(e)[:100]))
    
    # Checkpoint every 20 videos
    if (i+1) % 20 == 0:
        print(f'  [{i+1}/{len(videos_to_download)}] OK={len(downloaded)} FAIL={len(failed)}')

print(f'\n✅ Downloaded: {len(downloaded)}')
print(f'❌ Failed: {len(failed)}')
if failed:
    print('Sample failures:')
    for vid, err in failed[:5]:
        print(f'  {vid}: {err}')

## Cell 5: Parse StandUp4AI Labels

In [ ]:
def load_standup4ai_labels(repo_path, video_id, lang=None):
    """Load word-level labels for a video from StandUp4AI dataset."""
    labels = []
    
    # Try to find the video in laughter_detection/EMNLP/
    emnlp_dir = repo_path / 'laughter_detection' / 'EMNLP'
    
    # Search all language directories
    for ld_subdir in sorted(emnlp_dir.iterdir()) if emnlp_dir.exists() else []:
        if not ld_subdir.is_dir():
            continue
        for f in ld_subdir.glob(f'*{video_id}*'):
            try:
                if f.suffix == '.csv':
                    df = pd.read_csv(f)
                    return df
                elif f.suffix == '.json':
                    with open(f) as fh:
                        return json.load(fh)
            except:
                pass
    
    # Also check dataset directory
    dataset_dir = repo_path / 'dataset'
    for lang_dir in sorted(dataset_dir.iterdir()):
        if not lang_dir.is_dir():
            continue
        for f in lang_dir.rglob(f'*{video_id}*'):
            try:
                if f.suffix == '.csv':
                    df = pd.read_csv(f)
                    if len(df) > 0:
                        return df
            except:
                pass
    
    return None

# Test label loading
sample_vid = test_videos[0]
sample_labels = load_standup4ai_labels(repo_path, sample_vid)
if sample_labels is not None:
    print(f'Sample labels for {sample_vid}:')
    print(type(sample_labels))
    if hasattr(sample_labels, 'head'):
        print(sample_labels.head(10))
        print(f'Columns: {list(sample_labels.columns)}')
        print(f'Shape: {sample_labels.shape}')
    else:
        print(json.dumps(sample_labels, indent=2)[:500])
else:
    print(f'No labels found for {sample_vid}')
    # List what files exist in EMNLP
    emnlp_dir = repo_path / 'laughter_detection' / 'EMNLP'
    if emnlp_dir.exists():
        print(f'\nEMNLP dir contents:')
        for item in emnlp_dir.iterdir():
            print(f'  {item.name}')

## Cell 6: Extract F0 Features + Evaluate

In [ ]:
def extract_f0_features(audio_path, start, end, sr=22050):
    """Extract 5-dim F0 features for a segment."""
    try:
        duration = end - start
        if duration < 0.1:
            return None
        
        y, sr = librosa.load(audio_path, sr=sr, offset=start, duration=duration)
        if len(y) < sr * 0.1:  # Too short
            return None
        
        # Extract F0 via pyin
        f0, voiced, _ = librosa.pyin(
            y, fmin=50, fmax=500,
            frame_length=2048, hop_length=512,
            sr=sr
        )
        
        f0_voiced = f0[voiced & ~np.isnan(f0)] if len(f0) > 0 else np.array([])
        
        if len(f0_voiced) == 0:
            return np.array([0.0, 0.0, 0.0, 0.0, 0.0])
        
        voiced_rate = np.mean(voiced) if len(voiced) > 0 else 0.0
        
        features = np.array([
            np.mean(f0_voiced),
            np.std(f0_voiced),
            np.max(f0_voiced),
            np.min(f0_voiced),
            voiced_rate
        ])
        return features
    except Exception as e:
        return None

def extract_fast_features(audio_path, start, end, sr=22050):
    """Fast feature extraction using spectral features (no pyin)."""
    try:
        duration = end - start
        if duration < 0.1:
            return None
        
        y, sr = librosa.load(audio_path, sr=sr, offset=start, duration=min(duration, 10.0))
        if len(y) < sr * 0.1:
            return None
        
        # Fast features
        rms = librosa.feature.rms(y=y, hop_length=512)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=512)[0]
        
        features = np.array([
            np.mean(rms), np.std(rms), np.max(rms),
            np.mean(zcr), np.std(zcr),
            np.mean(centroid), np.std(centroid),
            np.mean(y**2),  # Energy
            len(y) / sr,  # Duration
        ])
        return features
    except:
        return None

print('Feature extraction functions ready')
print('F0 (5-dim): mean, std, max, min, voiced_rate')
print('Spectral (9-dim): rms, zcr, centroid, energy, duration')

## Cell 7: Run Full Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')

all_features = []
all_labels = []
all_videos = []
all_langs = []

# Process test videos that have both audio and labels
processed = 0
for vid in tqdm(test_videos):
    audio_path = BASE / 'audio' / f'{vid}.m4a'
    if not audio_path.exists():
        continue
    
    labels = load_standup4ai_labels(repo_path, vid)
    if labels is None:
        continue
    
    # Parse labels (format depends on file type)
    if hasattr(labels, 'values'):  # DataFrame
        for _, row in labels.iterrows():
            # Try different column names
            t0 = row.get('t0', row.get('start', row.get('timestamp_start', None)))
            t1 = row.get('t1', row.get('end', row.get('timestamp_end', None)))
            label = row.get('t', row.get('label', row.get('laughter', None)))
            
            if t0 is not None and t1 is not None and label is not None:
                feat = extract_fast_features(audio_path, float(t0), float(t1))
                if feat is not None:
                    all_features.append(feat)
                    all_labels.append(1 if str(label).strip() != 'O' and str(label) != '0' else 0)
                    all_videos.append(vid)
    
    processed += 1
    if processed % 10 == 0:
        X = np.array(all_features)
        y = np.array(all_labels)
        pos_rate = y.mean() if len(y) > 0 else 0
        print(f'  [{processed}] {len(y)} samples, {pos_rate:.1%} positive')

X = np.array(all_features)
y = np.array(all_labels)
videos = np.array(all_videos)

print(f'\n=== FINAL DATASET ===')
print(f'Samples: {len(y)}')
print(f'Positive: {y.sum()} ({y.mean():.1%})')
print(f'Videos: {len(set(videos))}')
print(f'Features: {X.shape[1]}-dim')

# Save features
np.savez_compressed(BASE / 'features' / 'standup4ai_features.npz',
                     X=X, y=y, videos=videos)
print(f'✅ Saved to Drive')

## Cell 8: Train + Evaluate (Video-Level Holdout)

In [ ]:
# Video-level cross-validation
gkf = GroupKFold(n_splits=min(5, len(set(videos))))
f1_scores = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, videos), 1):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    # Scale features
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    # Train Logistic Regression
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_tr_s, y_tr)
    
    # Predict
    y_pred = clf.predict(X_te_s)
    f1 = f1_score(y_te, y_pred, zero_division=0)
    f1_scores.append(f1)
    
    pos_rate = y_te.mean() if len(y_te) > 0 else 0
    print(f'Fold {fold}: F1={f1:.4f} (pos={pos_rate:.1%}, train={len(y_tr)}, test={len(y_te)})')

print(f'\n{"="*50}')
print(f'FINAL: F1={np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}')
print(f'StandUp4AI baseline: F1=0.51')
print(f'Our result:          F1={np.mean(f1_scores):.4f}')
if np.mean(f1_scores) > 0.51:
    print(f'🏆 WE BEAT THE BASELINE!')
else:
    print(f'⚠️ Below baseline. Need more features.')
print(f'{"="*50}')

## Cell 9: Per-Language Breakdown

In [ ]:
# Load partition to get language per video
vid_to_lang = dict(zip(partition['fn'], partition['lan']))
video_langs = [vid_to_lang.get(v, 'unknown') for v in videos]

# Per-language F1
from collections import defaultdict
lang_results = defaultdict(list)

# Use all data, predict with leave-one-video-out
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

unique_vids = list(set(videos))
for test_vid in unique_vids:
    test_mask = videos == test_vid
    train_mask = ~test_mask
    
    if y[test_mask].sum() == 0 and y[test_mask].sum() == len(y[test_mask]):
        continue  # Skip if all same class
    
    clf = LogisticRegression(max_iter=500, class_weight='balanced')
    clf.fit(X_s[train_mask], y[train_mask])
    pred = clf.predict(X_s[test_mask])
    
    lang = vid_to_lang.get(test_vid, 'unknown')
    lang_results[lang].append(f1_score(y[test_mask], pred, zero_division=0))

print('=== PER-LANGUAGE F1 ===')
print(f'{"Language":<12} {"F1":>8} {"Videos":>8}')
print('-' * 30)
for lang, scores in sorted(lang_results.items()):
    print(f'{lang:<12} {np.mean(scores):>8.4f} {len(scores):>8}')

print(f'\n{"="*50}')
all_scores = [s for scores in lang_results.values() for s in scores]
print(f'OVERALL: F1={np.mean(all_scores):.4f} ± {np.std(all_scores):.4f}')
print(f'StandUp4AI baseline: F1=0.51')
print(f'{"="*50}')

## Cell 10: Save Results + Summary

In [ ]:
results = {
    'experiment': 'standup4ai_f0_evaluation',
    'dataset': 'StandUp4AI test set',
    'features': f'{X.shape[1]}-dim spectral',
    'n_samples': len(y),
    'n_videos': len(set(videos)),
    'positive_rate': float(y.mean()),
    'f1_mean': float(np.mean(f1_scores)),
    'f1_std': float(np.std(f1_scores)),
    'standup4ai_baseline': 0.51,
    'beats_baseline': bool(np.mean(f1_scores) > 0.51),
    'per_language': {lang: float(np.mean(scores)) for lang, scores in lang_results.items()},
}

with open(BASE / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('=== FINAL RESULTS ===')
print(json.dumps(results, indent=2))
print(f'\n✅ Results saved to {BASE / "results.json"}')
print(f'✅ Features saved to {BASE / "features" / "standup4ai_features.npz"}')